In [1]:
from pathlib import Path

import pandas as pd

In [2]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [3]:
data = {
    "id": ["001", "002", "003", "004", "005", "006", "007", "008", "009"],
    "credential_list": [
        ["C01", "C02", "C03", "C04", "C05"],
        ["C01"],
        ["C02", "C04"],
        ["C01", "C05"],
        ["C02", "C03"],
        ["C01", "C02", "C03", "C04", "C05"],
        ["C03"],
        ["C05"],
        ["C01", "C04"],
    ],
    "provider_list": [
        ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10"],
        ["P02"],
        ["P01", "P04"],
        ["P03", "P06"],
        ["P02", "P07"],
        ["P08"],
        ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10"],
        ["P01"],
        ["P01"],
    ],
    "hotel_list": [
        ["H01"],
        ["H01"],
        ["H01", "H05"],
        ["H02", "H08"],
        ["H04", "H06", "H14"],
        ["H10", "H11"],
        ["H10", "H11"],
        ["H12", "H15"],
        ["H13"],
    ],
}

df = pd.DataFrame(data)
df["credential_count"] = df["credential_list"].apply(len)
df = df.sort_values(by="credential_count", ascending=False)

In [4]:
df_exploded = (
    df.explode("credential_list").explode("provider_list").explode("hotel_list")
)

df_exploded["combination_zip"] = list(
    zip(
        df_exploded["credential_list"],
        df_exploded["provider_list"],
        df_exploded["hotel_list"],
    )
)

In [5]:
df_exploded["is_duplicate"] = df_exploded.duplicated(
    subset="combination_zip", keep="first"
)

In [6]:
duplicate_ids = (
    df_exploded[df_exploded["is_duplicate"] | ~df_exploded["is_duplicate"]]
    .groupby("combination_zip")["id"]
    .apply(list)
    .reset_index(name="duplicate_ids")
)

In [7]:
df_exploded = df_exploded.merge(duplicate_ids, on="combination_zip", how="left")

In [8]:
duplicates = df_exploded[df_exploded["is_duplicate"]]

In [9]:
df_exploded.drop_duplicates(subset=["combination_zip"], keep="first", inplace=True)

In [10]:
df_exploded = (
    df_exploded.groupby("id")
    .agg(
        {
            "credential_list": lambda x: list(set(x)),
            "provider_list": lambda x: list(set(x)),
            "hotel_list": lambda x: list(set(x)),
            "duplicate_ids": "first",
        }
    )
    .reset_index()
)

In [11]:
duplicates = (
    duplicates.groupby("id")
    .agg(
        {
            "credential_list": lambda x: list(set(x)),
            "provider_list": lambda x: list(set(x)),
            "hotel_list": lambda x: list(set(x)),
            "duplicate_ids": "first",
        }
    )
    .reset_index()
)

In [12]:
df_exploded.head()

,id,credential_list,provider_list,hotel_list,duplicate_ids
0,001,"[C05, C01, C04, C03, C02]","[P10, P04, P03, P07, P08, P06, P09, P05, P01, ...",[H01],[001]
1,003,"[C04, C02]","[P04, P01]",[H05],[003]
2,004,"[C05, C01]","[P06, P03]","[H02, H08]",[004]
3,005,"[C02, C03]","[P07, P02]","[H06, H14, H04]",[005]
4,006,"[C05, C01, C04, C03, C02]",[P08],"[H10, H11]",[006]


In [13]:
duplicates.head()

,id,credential_list,provider_list,hotel_list,duplicate_ids
0,002,[C01],[P02],[H01],"[001, 002]"
1,003,"[C04, C02]","[P04, P01]",[H01],"[001, 003]"
2,007,[C03],[P08],"[H10, H11]","[006, 007]"


In [14]:
df.head()

,id,credential_list,provider_list,hotel_list,credential_count
0,001,"[C01, C02, C03, C04, C05]","[P01, P02, P03, P04, P05, P06, P07, P08, P09, ...",[H01],5
5,006,"[C01, C02, C03, C04, C05]",[P08],"[H10, H11]",5
2,003,"[C02, C04]","[P01, P04]","[H01, H05]",2
3,004,"[C01, C05]","[P03, P06]","[H02, H08]",2
4,005,"[C02, C03]","[P02, P07]","[H04, H06, H14]",2


In [15]:
df["updated_credential_list"] = df["id"].map(
    df_exploded.set_index("id")["credential_list"]
)
df["deleted_credential_list"] = df["id"].map(
    duplicates.set_index("id")["credential_list"]
)
df["updated_provider_list"] = df["id"].map(df_exploded.set_index("id")["provider_list"])
df["deleted_provider_list"] = df["id"].map(duplicates.set_index("id")["provider_list"])
df["updated_hotel_list"] = df["id"].map(df_exploded.set_index("id")["hotel_list"])
df["deleted_hotel_list"] = df["id"].map(duplicates.set_index("id")["hotel_list"])

df["duplicate_ids"] = df["id"].map(duplicates.set_index("id")["duplicate_ids"])

In [16]:
df.columns

Index(['id', 'credential_list', 'provider_list', 'hotel_list',
       'credential_count', 'updated_credential_list',
       'deleted_credential_list', 'updated_provider_list',
       'deleted_provider_list', 'updated_hotel_list', 'deleted_hotel_list',
       'duplicate_ids'],
      dtype='object')

In [17]:
df = df[
    [
        "id",
        "duplicate_ids",
        "credential_count",
        "credential_list",
        "updated_credential_list",
        "deleted_credential_list",
        "provider_list",
        "updated_provider_list",
        "deleted_provider_list",
        "hotel_list",
        "updated_hotel_list",
        "deleted_hotel_list",
    ]
]

In [18]:
df.to_csv("output.csv", index=False)

In [31]:
change_data = []

for index, row in df.iterrows():
    id_ = row["id"]

    # Original lists for this ID
    original_credential_list = row["credential_list"]
    original_provider_list = row["provider_list"]
    original_hotel_list = row["hotel_list"]

    # Get duplicate IDs for the current ID
    duplicate_ids = duplicates.loc[duplicates["id"] == id_, "duplicate_ids"].values
    duplicate_ids = duplicate_ids[0] if duplicate_ids.size > 0 else []

    # Updated lists after deletion (removing duplicates)
    updated_credential_list = (
        df_exploded.loc[df_exploded["id"] == id_, "credential_list"]
        .apply(lambda x: x if isinstance(x, list) else [])
        .explode()
        .unique()
        .tolist()
    )
    updated_provider_list = (
        df_exploded.loc[df_exploded["id"] == id_, "provider_list"]
        .apply(lambda x: x if isinstance(x, list) else [])
        .explode()
        .unique()
        .tolist()
    )
    updated_hotel_list = (
        df_exploded.loc[df_exploded["id"] == id_, "hotel_list"]
        .apply(lambda x: x if isinstance(x, list) else [])
        .explode()
        .unique()
        .tolist()
    )

    # Find deleted items
    deleted_credential_list = list(
        set(original_credential_list) - set(updated_credential_list)
    )
    deleted_provider_list = list(
        set(original_provider_list) - set(updated_provider_list)
    )
    deleted_hotel_list = list(set(original_hotel_list) - set(updated_hotel_list))

    # Count the credentials
    credential_count = len(updated_credential_list)

    # Append the results to the change data list
    change_data.append(
        {
            "id": id_,
            "duplicate_ids": duplicate_ids,
            "credential_count": credential_count,
            "credential_list": original_credential_list,
            "updated_credential_list": updated_credential_list,
            "deleted_credential_list": deleted_credential_list,
            "provider_list": original_provider_list,
            "updated_provider_list": updated_provider_list,
            "deleted_provider_list": deleted_provider_list,
            "hotel_list": original_hotel_list,
            "updated_hotel_list": updated_hotel_list,
            "deleted_hotel_list": deleted_hotel_list,
        }
    )

change_tracker = pd.DataFrame(change_data)

In [32]:
change_tracker.to_csv("output.csv", index=False)